# 05 — Persistent Memory: The Personal AI Assistant

Everything we've done so far loses memory when Python restarts.
This notebook builds **real long-term memory** — saved to a `.json` file on disk.

### What changes
```
Without persistence:  start Python → talk → close Python → memory gone ✗
With persistence:     start Python → talk → close Python → reopen  → memory back ✓
```

### What the agent remembers across sessions
- Your **name, age, city** — personal facts
- Your **preferences** — hobbies, favourite things
- The **last 10 messages** — recent context
- **When you last talked** — timestamps

### What you will learn
| Concept | Description |
|---------|-------------|
| `json` read/write | Save and load memory as a file |
| Auto-extraction | Parse name, age, hobbies from natural language |
| Personalised prompts | Use stored facts to build a rich system prompt |
| Multi-session | Each new conversation loads prior memory from disk |

> **Requires:** `GROQ_API_KEY` in a `.env` file

## 1. Install & Import

In [ ]:
# !pip install langchain-groq python-dotenv

In [ ]:
import os, json
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv()

llm = ChatGroq(model="llama3-8b-8192", temperature=0.7)
print("Ready!")

## 2. The PersistentMemory Class

In [ ]:
class PersistentMemory:
    """
    Saves agent memory to a JSON file.
    Loading the file restores memory from the previous session.
    """

    def __init__(self, filepath: str = "agent_memory.json"):
        self.filepath = Path(filepath)
        self.data     = self._load()

    # ── Load from file (or create empty memory) ───────────────────────────
    def _load(self) -> dict:
        if self.filepath.exists():
            with open(self.filepath, "r") as f:
                data = json.load(f)
            print(f"  [Memory] Loaded from '{self.filepath}'")
            return data
        else:
            print(f"  [Memory] No file found — starting fresh")
            return {
                "user": {},           # name, age, city, job…
                "preferences": [],    # hobbies, favourite things
                "history": [],        # recent messages
                "sessions": 0,        # how many times we've talked
                "last_seen": None
            }

    # ── Save to file ───────────────────────────────────────────────────────
    def save(self):
        with open(self.filepath, "w") as f:
            json.dump(self.data, f, indent=2)

    # ── Store a user fact ──────────────────────────────────────────────────
    def remember_fact(self, key: str, value):
        self.data["user"][key] = value
        self.save()
        print(f"  [Memory] Remembered: {key} = {value}")

    # ── Store a preference ─────────────────────────────────────────────────
    def add_preference(self, pref: str):
        if pref not in self.data["preferences"]:
            self.data["preferences"].append(pref)
            self.save()
            print(f"  [Memory] New preference noted: {pref}")

    # ── Add to conversation history (keep last 10 exchanges) ──────────────
    def add_to_history(self, role: str, message: str):
        entry = {
            "role":    role,
            "message": message,
            "time":    datetime.now().strftime("%Y-%m-%d %H:%M")
        }
        self.data["history"].append(entry)
        # Keep only the most recent 20 messages (10 exchanges)
        if len(self.data["history"]) > 20:
            self.data["history"] = self.data["history"][-20:]
        self.save()

    # ── Mark a new session ─────────────────────────────────────────────────
    def start_session(self):
        self.data["sessions"] += 1
        self.data["last_seen"] = datetime.now().strftime("%Y-%m-%d %H:%M")
        self.save()

    # ── Build context string for the LLM ──────────────────────────────────
    def build_context(self) -> str:
        lines = []

        if self.data["user"]:
            facts = ", ".join(f"{k}: {v}" for k, v in self.data["user"].items())
            lines.append(f"User facts: {facts}")

        if self.data["preferences"]:
            prefs = ", ".join(self.data["preferences"])
            lines.append(f"User preferences: {prefs}")

        if self.data["sessions"] > 1:
            lines.append(f"This is session #{self.data['sessions']}. Last seen: {self.data['last_seen']}")

        if self.data["history"]:
            lines.append("Recent messages:")
            for entry in self.data["history"][-8:]:
                label = "User" if entry["role"] == "user" else "Assistant"
                lines.append(f"  {label}: {entry['message']}")

        return "\n".join(lines)

    def show(self):
        print("\n── Memory File Contents ──────────────────────────────")
        print(json.dumps(self.data, indent=2)[:1200])
        print("─────────────────────────────────────────────────────\n")

print("PersistentMemory class defined!")

## 3. Auto-Extraction of Facts

In [ ]:
def extract_and_remember(text: str, memory: PersistentMemory):
    """
    Parse simple facts out of a message and store them in memory.
    No ML needed — just string matching.
    """
    t = text.lower()

    # Extract name
    for pattern in ("my name is ", "i'm ", "i am ", "call me "):
        if pattern in t:
            words = t.split(pattern, 1)[-1].strip().split()
            if words and words[0].isalpha():
                memory.remember_fact("name", words[0].title())
                break

    # Extract age
    for pattern in ("i am ", "i'm "):
        if pattern in t and "year" in t:
            words = t.split(pattern, 1)[-1].strip().split()
            if words and words[0].isdigit():
                memory.remember_fact("age", int(words[0]))
                break

    # Extract city / location
    for pattern in ("from ", "live in ", "based in ", "in "):
        if pattern in t and "from" in t:
            part = t.split("from ", 1)[-1].strip().split()[0]
            if part.isalpha() and len(part) > 2:
                memory.remember_fact("city", part.title())
                break

    # Extract job / role
    for pattern in ("i work as ", "i'm a ", "i am a "):
        if pattern in t:
            part = t.split(pattern, 1)[-1].strip()
            job  = " ".join(w for w in part.split()[:3] if w.isalpha())
            if job:
                memory.remember_fact("job", job.title())
                break

    # Extract preferences
    for pattern in ("i love ", "i like ", "i enjoy ", "my favourite "):
        if pattern in t:
            pref = t.split(pattern, 1)[-1].strip().split(".")[0]
            memory.add_preference(pref[:60])   # cap length

print("Auto-extraction helper ready!")

## 4. Build the Personalised Agent

In [ ]:
def personal_agent(user_input: str, memory: PersistentMemory) -> str:
    """
    Chat function that:
    1. Builds a rich system prompt from stored memory
    2. Calls the LLM with that context
    3. Extracts new facts from the user's message
    4. Saves everything to disk
    """
    # ── Build the system prompt ──────────────────────────────────────────
    context = memory.build_context()
    system_content = (
        "You are a warm, personalised AI assistant who remembers everything "
        "about the user across conversations.\n"
        "Always address the user by name if you know it. "
        "Refer to things they told you in earlier sessions to show you remember.\n"
    )
    if context:
        system_content += f"\nWhat you already know:\n{context}"

    # ── Call the LLM ────────────────────────────────────────────────────
    response = llm.invoke([
        SystemMessage(content=system_content),
        HumanMessage(content=user_input),
    ])
    reply = response.content.strip()

    # ── Extract & save facts ─────────────────────────────────────────────
    extract_and_remember(user_input, memory)
    memory.add_to_history("user",      user_input)
    memory.add_to_history("assistant", reply)

    print(f"  You : {user_input}")
    print(f"  Bot : {reply}\n")
    return reply

print("Personal agent ready!")

## 5. Session 1 — Meet the Agent

In [ ]:
# ── Session 1 ────────────────────────────────────────────────────────────
print("=" * 52)
print("  SESSION 1 — First conversation")
print("=" * 52 + "\n")

# Create a fresh memory (or load from file if it already exists)
mem = PersistentMemory("agent_memory.json")
mem.start_session()

personal_agent("Hello! My name is Vikram.", mem)
personal_agent("I am 30 years old and from Hyderabad.", mem)
personal_agent("I work as a machine learning engineer.", mem)
personal_agent("I love cricket and reading sci-fi books.", mem)
personal_agent("What do you know about me so far?", mem)

## 6. Inspect the Memory File

In [ ]:
# Let's look at what got saved to disk
mem.show()

## 7. Session 2 — Restart and Resume

The cell below simulates **closing and reopening Python**.
It creates a brand-new `PersistentMemory` object, which immediately
reads the file — loading everything from Session 1.

In [ ]:
# ── Session 2 — load memory from disk ────────────────────────────────────
print("=" * 52)
print("  SESSION 2 — After restart (loading from file)")
print("=" * 52 + "\n")

# Create NEW object → loads from file automatically
mem2 = PersistentMemory("agent_memory.json")
mem2.start_session()

# These should feel personalised without re-introducing yourself
personal_agent("Hey, do you remember me?",         mem2)
personal_agent("What city am I from?",             mem2)
personal_agent("What do I do for work?",           mem2)
personal_agent("Recommend a sci-fi book for me.",  mem2)

## 8. Clear Memory When Needed

In [ ]:
def clear_memory(filepath: str = "agent_memory.json"):
    """Delete the memory file to start completely fresh."""    p = Path(filepath)
    if p.exists():
        p.unlink()
        print(f"Memory file '{filepath}' deleted. Starting fresh next session.")
    else:
        print("No memory file found — already clean.")

# Uncomment the line below to wipe all memory:
# clear_memory("agent_memory.json")
print("clear_memory() is available — uncomment to reset.")

## 9. Full Memory & State Summary

You've now learned every major type of memory for AI agents:

| Notebook | Memory Type | Persists Restart? | Best For |
|----------|-------------|------------------|----------|
| 01 | In-memory dict | ❌ No | Understanding the concept |
| 02 | Buffer (message list) | ❌ No | Short conversations |
| 03 | Summary (LLM compression) | ❌ No | Long conversations, token saving |
| 04 | LangGraph MemorySaver | ❌ No | Multi-user sessions in one app |
| 05 | JSON file (persistent) | ✅ Yes | Personal assistants, long-term agents |

### Real production systems combine all of these:
```
Short-term  → Window/Summary buffer for the current conversation
Long-term   → Database (SQLite / PostgreSQL) for facts and history
Session     → Thread IDs to separate users
```

**Congratulations — you now understand Memory & State for AI agents! 🎉**